Preparo dos dados e verificação da configuração

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("imdb_spark").getOrCreate()

df_titles = spark.read.csv("title_basics.tsv", header=True, inferSchema=True, sep="\t", nullValue="\\N")
df_ratings = spark.read.csv("title_ratings.tsv", header=True, inferSchema=True, sep="\t", nullValue="\\N")

df = df_titles.join(df_ratings, on="tconst", how="inner")

df.printSchema()
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/06 11:18:09 WARN Utils: Your hostname, ViniWenz, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/06 11:18:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/06 11:18:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- tconst: string (nullable = true)
 |-- titleType: string (nullable = true)
 |-- primaryTitle: string (nullable = true)
 |-- originalTitle: string (nullable = true)
 |-- isAdult: integer (nullable = true)
 |-- startYear: integer (nullable = true)
 |-- endYear: integer (nullable = true)
 |-- runtimeMinutes: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- averageRating: double (nullable = true)
 |-- numVotes: integer (nullable = true)



+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+-----------------+-------------+--------+
|   tconst|titleType|        primaryTitle|       originalTitle|isAdult|startYear|endYear|runtimeMinutes|           genres|averageRating|numVotes|
+---------+---------+--------------------+--------------------+-------+---------+-------+--------------+-----------------+-------------+--------+
|tt0000002|    short|Le clown et ses c...|Le clown et ses c...|      0|     1892|   NULL|             5|  Animation,Short|          6.0|     233|
|tt0000004|    short|         Un bon bock|         Un bon bock|      0|     1892|   NULL|            12|  Animation,Short|          6.1|     152|
|tt0000008|    short|Edison Kinetoscop...|Edison Kinetoscop...|      0|     1894|   NULL|             1|Documentary,Short|          5.5|    1965|
|tt0000015|    short| Autour d'une cabine| Autour d'une cabine|      0|     1894|   NULL|             2|  Animation,Short|  

Quantos filmes (incluindo os da televisão) foram lançados no ano de 2015?

In [2]:
# conferindo os possíveis valores de tilteType primeiro
df_titles.select("titleType").distinct().show()

+------------+
|   titleType|
+------------+
|    tvSeries|
|tvMiniSeries|
|     tvMovie|
|   tvEpisode|
|       movie|
|   tvSpecial|
|       video|
|   videoGame|
|     tvShort|
|       short|
|     tvPilot|
| radioSeries|
|radioEpisode|
+------------+



In [3]:
df_titles.filter(
    (F.col("startYear") == 2015) &
    (F.col("titleType").isin("movie", "tvMovie"))
).count()

19987

Qual o gênero de títulos mais frequente?

In [5]:
# após o df.show(5) do início, vemos que existem mais de um gênero em alguns casos na coluna, separados por vírgula

df_titles.withColumn("genre", F.explode(F.split(F.col("genres"), ","))) \
    .groupBy("genre") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+-----------+-------+
|      genre|  count|
+-----------+-------+
|      Drama|2247995|
|     Comedy|1653725|
|      Short|1021850|
|  Talk-Show| 900198|
|Documentary| 764885|
|    Romance| 724729|
|     Family| 571470|
|       News| 524662|
| Reality-TV| 423455|
|  Animation| 406284|
|      Music| 394008|
|      Crime| 351447|
|     Action| 334580|
|  Adventure| 324325|
|  Game-Show| 252533|
|      Adult| 242704|
|      Sport| 178594|
|    Fantasy| 174119|
|    Mystery| 162448|
|     Horror| 146400|
+-----------+-------+
only showing top 20 rows


Qual o gênero com a melhor nota média de títulos?

In [ ]:
df.withColumn("genre", F.explode(F.split(F.col("genres"), ","))) \
    .groupBy("genre") \
    .agg(F.avg("averageRating").alias("genreAverageRating")) \
    .orderBy(F.desc("genreAverageRating")) \
    .show()

{"ts": "2026-09-06 11:41:16.529", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `genreAverageRating` cannot be resolved. Did you mean one of the following? [`avg(averageRating)`, `genre`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "103)", "fragment": "desc", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o132.sort.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `genreAverageRating` cannot be resolved. Did you mean one of the following? [`avg(averageRating)`, `genre`]. SQLSTATE: 42703;\n'Sort ['genreAverageRating DESC NULLS LAST], true\n+- SubqueryAlias genreAverageRating\n   +- Aggregate [genre#155], [genre#155

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `genreAverageRating` cannot be resolved. Did you mean one of the following? [`avg(averageRating)`, `genre`]. SQLSTATE: 42703;
'Sort ['genreAverageRating DESC NULLS LAST], true
+- SubqueryAlias genreAverageRating
   +- Aggregate [genre#155], [genre#155, avg(averageRating#44) AS avg(averageRating)#169]
      +- Project [tconst#17, titleType#18, primaryTitle#19, originalTitle#20, isAdult#21, startYear#22, endYear#23, runtimeMinutes#24, genres#25, averageRating#44, numVotes#45, genre#155]
         +- Generate explode(split(genres#25, ,, -1)), false, [genre#155]
            +- Project [tconst#17, titleType#18, primaryTitle#19, originalTitle#20, isAdult#21, startYear#22, endYear#23, runtimeMinutes#24, genres#25, averageRating#44, numVotes#45]
               +- Join Inner, (tconst#17 = tconst#43)
                  :- Relation [tconst#17,titleType#18,primaryTitle#19,originalTitle#20,isAdu